# 🧠 Taller de Ingeniería Biomédica con NERO
## IEEE EMBS × Universidad Tecnológica de Panamá

**Pipeline completo:** Simulación de biosignales → Procesamiento EMG → Clasificación → Control de Arduino vía PySerial

---

| Dato | Detalle |
|------|---------|
| Librería principal | NeuroKit2 + PySerial |
| Hardware opcional | BioAmp EXG Pill + Arduino Uno |
| Señales cubiertas | EMG · ECG · EEG (simuladas) |
| Aplicación final | Activación de servomotor (exoesqueleto NERO) |

> **Nota pedagógica:** Las secciones 1–4 funcionan completamente sin hardware.  
> La sección 5 requiere el BioAmp EXG Pill y el Arduino Uno conectados por USB.


---
## Sección 0 — Instalación de dependencias

Ejecuta esta celda **una sola vez** al inicio del taller.  
Si usas **Google Colab**, también funciona sin cambios.


In [ ]:
%pip install neurokit2 pyserial matplotlib numpy scipy --quiet
print("✅ Dependencias instaladas correctamente")


---
## Sección 1 — ¿Qué es una señal biológica?

Las señales bioeléctricas son diferencias de potencial generadas por actividad celular.  
En este bloque simulamos las tres más relevantes para sistemas BCI:

| Señal | Origen | Rango frecuencia | Amplitud típica |
|-------|--------|-----------------|-----------------|
| **EMG** | Contracción muscular | 20 – 500 Hz | 0.1 – 5 mV |
| **ECG** | Actividad cardíaca | 0.5 – 40 Hz | 1 – 5 mV |
| **EEG** | Actividad cerebral | 0.5 – 80 Hz | 10 – 100 µV |

> 💡 **Para pensar:** ¿Por qué el EMG tiene mayor rango de frecuencia que el ECG?


In [ ]:
import neurokit2 as nk
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Parámetros de simulación ──────────────────────────────────────────────
DURACION   = 5       # segundos
FS         = 1000    # frecuencia de muestreo (Hz)
HEART_RATE = 72      # latidos por minuto

# ── Simulación de señales ────────────────────────────────────────────────
emg = nk.emg_simulate(duration=DURACION, sampling_rate=FS, burst_number=4, burst_duration=0.3)
ecg = nk.ecg_simulate(duration=DURACION, sampling_rate=FS, heart_rate=HEART_RATE)
eeg = nk.signal_simulate(duration=DURACION, sampling_rate=FS, frequency=[8, 10, 13], noise=0.3)

tiempo = np.linspace(0, DURACION, len(emg))

# ── Visualización ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
fig.suptitle("Señales Biológicas Simuladas", fontsize=14, fontweight='bold')

configs = [
    (emg, "EMG — Electromiografía", "#E74C3C", "Amplitud (mV)"),
    (ecg, "ECG — Electrocardiografía", "#2980B9", "Amplitud (mV)"),
    (eeg, "EEG — Electroencefalografía (banda alfa)", "#27AE60", "Amplitud (µV)"),
]

for ax, (signal, titulo, color, ylabel) in zip(axes, configs):
    ax.plot(tiempo, signal, color=color, linewidth=0.8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(titulo, fontsize=10, fontweight='bold', loc='left')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel("Tiempo (s)")
plt.tight_layout()
plt.show()

print(f"\nMuestras generadas: {len(emg):,} por señal @ {FS} Hz")
print(f"Resolución temporal: {1000/FS:.1f} ms por muestra")


---
## Sección 2 — Pipeline de procesamiento EMG

Este es el pipeline exacto que implementa NERO para decidir si activar el exoesqueleto.

```
Señal cruda → Filtrado pasa-banda → Rectificación → Envolvente RMS → Umbral → Comando
```

Cada paso tiene un propósito fisiológico específico:

1. **Filtrado (20–500 Hz):** elimina ruido de red (60 Hz), artefactos de movimiento y DC offset
2. **Rectificación:** convierte valores negativos a positivos (la energía muscular no tiene signo)
3. **Envolvente RMS:** suaviza la señal para obtener la "intensidad" de contracción
4. **Umbral:** punto de decisión — ¿el músculo está activo o en reposo?


In [ ]:
from scipy.signal import butter, filtfilt
import warnings
warnings.filterwarnings('ignore')

# ── Parámetros del pipeline (igual que NERO) ──────────────────────────────
F_LOW      = 20      # Hz — frecuencia de corte inferior
F_HIGH     = 499     # Hz — frecuencia de corte superior
VENTANA_MS = 200     # ms — tamaño de ventana para RMS
UMBRAL     = 0.08    # V — umbral de activación (ajustable)

# ── Simular señal con reposo y contracción ────────────────────────────────
np.random.seed(42)
emg_raw = nk.emg_simulate(duration=6, sampling_rate=FS, burst_number=3,
                            burst_duration=0.6, noise=0.01)

# ── PASO 1: Filtrado pasa-banda ───────────────────────────────────────────
def filtro_pasabanda(señal, low, high, fs, orden=4):
    nyq = fs / 2
    b, a = butter(orden, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, señal)

emg_filtrado = filtro_pasabanda(emg_raw, F_LOW, F_HIGH, FS)

# ── PASO 2: Rectificación de onda completa ────────────────────────────────
emg_rectificado = np.abs(emg_filtrado)

# ── PASO 3: Envolvente RMS ────────────────────────────────────────────────
def calcular_rms(señal, ventana_muestras):
    rms = np.zeros(len(señal))
    mitad = ventana_muestras // 2
    for i in range(len(señal)):
        inicio = max(0, i - mitad)
        fin    = min(len(señal), i + mitad)
        rms[i] = np.sqrt(np.mean(señal[inicio:fin]**2))
    return rms

ventana_muestras = int(VENTANA_MS * FS / 1000)
emg_rms = calcular_rms(emg_rectificado, ventana_muestras)

# ── PASO 4: Clasificación por umbral ─────────────────────────────────────
activacion = (emg_rms > UMBRAL).astype(int)

# ── Visualización del pipeline completo ──────────────────────────────────
tiempo = np.linspace(0, 6, len(emg_raw))
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
fig.suptitle("Pipeline EMG — Igual al sistema NERO", fontsize=14, fontweight='bold')

pasos = [
    (emg_raw,       "1. Señal cruda (sin filtrar)",      "#95A5A6", False),
    (emg_filtrado,  "2. Filtrado pasa-banda (20–499 Hz)","#E67E22", False),
    (emg_rms,       "3. Envolvente RMS (200 ms)",        "#8E44AD", True),
    (activacion,    "4. Activación (umbral = {:.2f})".format(UMBRAL), "#27AE60", False),
]

for ax, (señal, titulo, color, mostrar_umbral) in zip(axes, pasos):
    ax.plot(tiempo, señal, color=color, linewidth=0.8)
    if mostrar_umbral:
        ax.axhline(y=UMBRAL, color='red', linestyle='--', linewidth=1.2, label=f'Umbral = {UMBRAL}')
        ax.fill_between(tiempo, señal, UMBRAL,
                        where=(señal > UMBRAL), alpha=0.25, color='red', label='Activación')
        ax.legend(fontsize=8, loc='upper right')
    ax.set_title(titulo, fontsize=9, fontweight='bold', loc='left')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel("Tiempo (s)")
plt.tight_layout()
plt.show()

# ── Resumen de la clasificación ───────────────────────────────────────────
n_activas = np.sum(activacion)
pct_activo = 100 * n_activas / len(activacion)
print(f"\n📊 Resumen del análisis:")
print(f"   Muestras totales : {len(activacion):,}")
print(f"   Muestras activas : {n_activas:,} ({pct_activo:.1f}%)")
print(f"   Valor RMS máximo : {emg_rms.max():.4f}")
print(f"   Umbral actual    : {UMBRAL}")
print(f"   Estado final     : {'🟢 ACTIVO' if activacion[-1] else '🔴 REPOSO'}")


---
## Sección 3 — Análisis ECG avanzado: detección de picos R y HRV

El ECG es más complejo que el EMG porque tiene estructura temporal (complejo PQRST).  
NeuroKit2 detecta automáticamente los picos R y calcula métricas clínicas reales.

**Variabilidad de Frecuencia Cardíaca (HRV):** mide las variaciones entre latidos consecutivos.  
Es un indicador del balance del sistema nervioso autónomo — usado en investigación clínica.


In [ ]:
# ── Simular y procesar ECG ────────────────────────────────────────────────
ecg_sim = nk.ecg_simulate(duration=30, sampling_rate=FS, heart_rate=72, noise=0.05)
señales_ecg, info_ecg = nk.ecg_process(ecg_sim, sampling_rate=FS)

# ── Visualización con picos R marcados ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle("Análisis ECG — Detección de picos R y HRV", fontsize=14, fontweight='bold')

t = np.linspace(0, 30, len(ecg_sim))
# Mostrar solo 5 segundos para mejor visualización
mask = t <= 5
ax = axes[0]
ax.plot(t[mask], ecg_sim[mask], color='#2980B9', linewidth=0.9, label='ECG simulado')

# Marcar picos R dentro de la ventana
picos_r = info_ecg["ECG_R_Peaks"]
picos_en_ventana = picos_r[picos_r < int(5 * FS)]
ax.scatter(t[picos_en_ventana], ecg_sim[picos_en_ventana],
           color='red', zorder=5, s=50, label='Picos R detectados')

ax.set_title("ECG — Primeros 5 segundos con picos R", fontsize=10, fontweight='bold', loc='left')
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud (mV)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Intervalos RR ─────────────────────────────────────────────────────────
rr_intervals = np.diff(picos_r) / FS * 1000  # en milisegundos
ax2 = axes[1]
ax2.bar(range(len(rr_intervals)), rr_intervals, color='#3498DB', alpha=0.7, width=0.8)
ax2.axhline(y=np.mean(rr_intervals), color='red', linestyle='--',
            linewidth=1.5, label=f'Media RR = {np.mean(rr_intervals):.0f} ms')
ax2.set_title("Intervalos RR (variabilidad entre latidos)", fontsize=10, fontweight='bold', loc='left')
ax2.set_xlabel("Número de latido")
ax2.set_ylabel("Intervalo RR (ms)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ── Métricas HRV ─────────────────────────────────────────────────────────
fc_media   = 60000 / np.mean(rr_intervals)
rmssd      = np.sqrt(np.mean(np.diff(rr_intervals)**2))
sdnn       = np.std(rr_intervals)

print(f"\n💓 Métricas cardiovasculares:")
print(f"   Frecuencia cardíaca media : {fc_media:.1f} lpm")
print(f"   Intervalo RR medio        : {np.mean(rr_intervals):.1f} ms")
print(f"   RMSSD (HRV dominio tiempo): {rmssd:.2f} ms")
print(f"   SDNN                      : {sdnn:.2f} ms")
print(f"   Latidos detectados        : {len(picos_r)}")
print(f"\n📌 Referencia clínica: RMSSD > 40 ms indica buena regulación autonómica")


---
## Sección 4 — Clasificador en tiempo real (simulado)

Aquí implementamos la lógica exacta que NERO usa para decidir si mover el exoesqueleto.

El clasificador recibe ventanas de señal EMG cada 100 ms y decide:
- `REPOSO` → no hacer nada
- `CONTRACCIÓN LEVE` → mover al 50%
- `CONTRACCIÓN FUERTE` → mover al 100%

> 🔬 En NERO real, este bloque corre en el microcontrolador. Aquí lo ejecutamos en Python  
> para visualizar la lógica antes de implementarla en Arduino.


In [ ]:
# ── Simulador de ventana deslizante ──────────────────────────────────────
VENTANA_MS     = 200    # ms por ventana
PASO_MS        = 100    # ms — overlap del 50%
UMBRAL_LEVE    = 0.05
UMBRAL_FUERTE  = 0.12

ventana_n = int(VENTANA_MS * FS / 1000)
paso_n    = int(PASO_MS   * FS / 1000)

# Señal de prueba más larga (10 s con 4 contracciones)
emg_test = nk.emg_simulate(duration=10, sampling_rate=FS, burst_number=4,
                             burst_duration=0.8, noise=0.008)
emg_test_filt = filtro_pasabanda(emg_test, F_LOW, F_HIGH, FS)

# ── Clasificación ventana a ventana ──────────────────────────────────────
tiempos_ventana = []
rms_ventanas    = []
clases          = []
comandos_servo  = []

for inicio in range(0, len(emg_test_filt) - ventana_n, paso_n):
    fin      = inicio + ventana_n
    ventana  = emg_test_filt[inicio:fin]
    rms_val  = np.sqrt(np.mean(ventana**2))
    t_centro = (inicio + fin) / 2 / FS

    if rms_val < UMBRAL_LEVE:
        clase   = "REPOSO"
        servo   = 0
    elif rms_val < UMBRAL_FUERTE:
        clase   = "LEVE"
        servo   = 50
    else:
        clase   = "FUERTE"
        servo   = 100

    tiempos_ventana.append(t_centro)
    rms_ventanas.append(rms_val)
    clases.append(clase)
    comandos_servo.append(servo)

# ── Visualización del clasificador ───────────────────────────────────────
colores_clase = {"REPOSO": "#95A5A6", "LEVE": "#F39C12", "FUERTE": "#E74C3C"}

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle("Clasificador NERO — Ventana deslizante 200 ms / paso 100 ms",
             fontsize=13, fontweight='bold')

t_full = np.linspace(0, 10, len(emg_test))

# Señal EMG filtrada
axes[0].plot(t_full, emg_test_filt, color='#E74C3C', linewidth=0.7, alpha=0.8)
axes[0].set_title("Señal EMG filtrada (20–499 Hz)", fontsize=9, fontweight='bold', loc='left')
axes[0].set_ylabel("Amplitud (V)")
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# RMS por ventana + umbrales
axes[1].plot(tiempos_ventana, rms_ventanas, color='#8E44AD', linewidth=1.2, label='RMS por ventana')
axes[1].axhline(UMBRAL_LEVE,   color='#F39C12', linestyle='--', linewidth=1, label=f'Umbral leve = {UMBRAL_LEVE}')
axes[1].axhline(UMBRAL_FUERTE, color='#E74C3C', linestyle='--', linewidth=1, label=f'Umbral fuerte = {UMBRAL_FUERTE}')
axes[1].fill_between(tiempos_ventana, rms_ventanas, UMBRAL_FUERTE,
                     where=[r > UMBRAL_FUERTE for r in rms_ventanas],
                     alpha=0.3, color='#E74C3C')
axes[1].fill_between(tiempos_ventana, rms_ventanas, UMBRAL_LEVE,
                     where=[(UMBRAL_LEVE < r <= UMBRAL_FUERTE) for r in rms_ventanas],
                     alpha=0.3, color='#F39C12')
axes[1].set_title("Envolvente RMS con zonas de activación", fontsize=9, fontweight='bold', loc='left')
axes[1].set_ylabel("RMS (V)")
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# Comando al servo
colores_barra = [colores_clase[c] for c in clases]
axes[2].bar(tiempos_ventana, comandos_servo, width=0.09,
            color=colores_barra, alpha=0.85, label='Posición servo (°)')
axes[2].set_yticks([0, 50, 100])
axes[2].set_yticklabels(['0° — Reposo', '50° — Leve', '100° — Fuerte'])
axes[2].set_title("Comando enviado al servomotor (exoesqueleto)", fontsize=9, fontweight='bold', loc='left')
axes[2].set_xlabel("Tiempo (s)")
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ── Estadísticas de clasificación ────────────────────────────────────────
from collections import Counter
conteo = Counter(clases)
total  = len(clases)
print("\n📊 Distribución de clasificaciones:")
for clase, n in sorted(conteo.items()):
    print(f"   {clase:10s}: {n:3d} ventanas ({100*n/total:.1f}%)")
print(f"\n   Total ventanas analizadas: {total}")
print(f"   Tiempo de decisión por ventana: {PASO_MS} ms")


---
## Sección 5 — Conexión real: PySerial → Arduino Uno

> ⚠️ **Esta sección requiere hardware físico:**
> - Arduino Uno conectado por USB
> - BioAmp EXG Pill conectado al pin A0 del Arduino
> - Electrodos Ag/AgCl colocados en el antebrazo/bíceps

### Protocolo de comunicación

Python envía **un byte** al Arduino que representa el ángulo del servo:
- `b'0'` → 0° (reposo)
- `b'5'` → 50° (contracción leve)  
- `b'1'` → 100° (contracción fuerte)

### Código para cargar en el Arduino Uno

```cpp
#include <Servo.h>

Servo miservo;
const int SERVO_PIN = 9;
const int EMG_PIN   = A0;

void setup() {
  Serial.begin(115200);
  miservo.attach(SERVO_PIN);
  miservo.write(0);
  Serial.println("NERO_READY");
}

void loop() {
  // ── Leer señal EMG desde BioAmp EXG Pill ──
  int emg_raw = analogRead(EMG_PIN);
  Serial.println(emg_raw);   // enviar dato a Python

  // ── Recibir comando de Python ──
  if (Serial.available() > 0) {
    char cmd = Serial.read();
    if      (cmd == '0') miservo.write(0);
    else if (cmd == '5') miservo.write(50);
    else if (cmd == '1') miservo.write(100);
  }

  delay(10);  // 100 Hz de muestreo
}
```


In [ ]:
import serial
import serial.tools.list_ports
import time
import threading
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# ── Detectar puertos disponibles ──────────────────────────────────────────
print("🔍 Puertos seriales disponibles:")
puertos = serial.tools.list_ports.comports()
if puertos:
    for p in puertos:
        print(f"   {p.device:15s} — {p.description}")
else:
    print("   ⚠️  No se detectaron puertos. Verifica la conexión USB del Arduino.")

# ── Configuración (cambiar según tu sistema) ──────────────────────────────
# Windows: 'COM3', 'COM4', etc.
# Linux/Mac: '/dev/ttyUSB0', '/dev/ttyACM0'
PUERTO_ARDUINO = 'COM3'       # ← CAMBIAR según tu sistema
BAUD_RATE      = 115200
FS_ARDUINO     = 100          # Hz (delay(10) en Arduino)
BUFFER_SIZE    = 200          # muestras en el buffer circular

print(f"\n⚙️  Configuración: Puerto={PUERTO_ARDUINO}, Baudrate={BAUD_RATE}, Fs={FS_ARDUINO} Hz")


In [ ]:
# ── Clase principal: BioAmpNERO ──────────────────────────────────────────
# Maneja lectura EMG, procesamiento y envío de comandos en hilos separados.

class BioAmpNERO:
    """
    Interfaz Python ↔ Arduino para el sistema NERO.
    
    Uso:
        nero = BioAmpNERO(puerto='COM3')
        nero.iniciar()
        time.sleep(10)  # capturar 10 segundos
        nero.detener()
        nero.graficar()
    """

    def __init__(self, puerto, baud=115200, fs=100,
                 umbral_leve=0.05, umbral_fuerte=0.12,
                 ventana_ms=200):
        self.puerto         = puerto
        self.baud           = baud
        self.fs             = fs
        self.umbral_leve    = umbral_leve
        self.umbral_fuerte  = umbral_fuerte
        self.ventana_n      = int(ventana_ms * fs / 1000)

        self.buffer_raw     = deque(maxlen=BUFFER_SIZE)
        self.buffer_tiempo  = deque(maxlen=BUFFER_SIZE)
        self.buffer_rms     = deque(maxlen=BUFFER_SIZE)
        self.buffer_cmds    = deque(maxlen=BUFFER_SIZE)

        self.ser            = None
        self._activo        = False
        self._t_inicio      = None
        self._hilo_lectura  = None
        self._ultimo_cmd    = '0'

    def _voltaje(self, adc_val, vref=5.0, bits=10):
        """Convierte valor ADC a voltaje (Arduino 10-bit, 5V)."""
        return (adc_val / (2**bits - 1)) * vref - vref / 2

    def _calcular_rms(self):
        """RMS de las últimas ventana_n muestras del buffer."""
        if len(self.buffer_raw) < self.ventana_n:
            return 0.0
        ventana = list(self.buffer_raw)[-self.ventana_n:]
        return float(np.sqrt(np.mean(np.array(ventana)**2)))

    def _clasificar(self, rms_val):
        """Regresa comando ('0','5','1') y etiqueta."""
        if rms_val >= self.umbral_fuerte:
            return '1', 'FUERTE'
        elif rms_val >= self.umbral_leve:
            return '5', 'LEVE'
        else:
            return '0', 'REPOSO'

    def _loop_lectura(self):
        """Hilo de lectura: lee EMG, procesa y envía comando al Arduino."""
        while self._activo:
            try:
                linea = self.ser.readline().decode('utf-8', errors='ignore').strip()
                if not linea or not linea.isdigit():
                    continue

                adc_val = int(linea)
                voltaje = self._voltaje(adc_val)
                t_actual = time.time() - self._t_inicio

                # Aplicar filtro básico (pasa-alto simple) para quitar DC
                if self.buffer_raw:
                    voltaje = voltaje - 0.95 * (list(self.buffer_raw)[-1])

                self.buffer_raw.append(voltaje)
                self.buffer_tiempo.append(t_actual)

                rms_val    = self._calcular_rms()
                cmd, label = self._clasificar(rms_val)

                self.buffer_rms.append(rms_val)
                self.buffer_cmds.append(label)

                # Enviar comando solo si cambia (evita saturar el bus serial)
                if cmd != self._ultimo_cmd:
                    self.ser.write(cmd.encode())
                    self._ultimo_cmd = cmd
                    print(f"  [{t_actual:6.2f}s] RMS={rms_val:.4f}V → {label} → Servo {cmd}")

            except Exception as e:
                if self._activo:
                    print(f"  ⚠️ Error de lectura: {e}")

    def iniciar(self):
        """Abre el puerto serial e inicia el hilo de lectura."""
        try:
            self.ser = serial.Serial(self.puerto, self.baud, timeout=1)
            time.sleep(2)  # esperar reset del Arduino
            self.ser.flushInput()
            self._activo  = True
            self._t_inicio = time.time()
            self._hilo_lectura = threading.Thread(target=self._loop_lectura, daemon=True)
            self._hilo_lectura.start()
            print(f"✅ Conectado a {self.puerto} @ {self.baud} baud")
            print(f"   Umbrales → Leve: {self.umbral_leve} V | Fuerte: {self.umbral_fuerte} V")
            print(f"   Ventana RMS: {self.ventana_n} muestras ({self.ventana_n/self.fs*1000:.0f} ms)")
            print("\n🟢 Capturando señal EMG... (ejecuta nero.detener() para parar)\n")
        except serial.SerialException as e:
            print(f"❌ No se pudo abrir {self.puerto}: {e}")
            print("   Verifica: 1) Arduino conectado  2) Puerto correcto  3) Arduino IDE cerrado")

    def detener(self):
        """Detiene la captura y cierra el puerto."""
        self._activo = False
        if self._hilo_lectura:
            self._hilo_lectura.join(timeout=2)
        if self.ser and self.ser.is_open:
            self.ser.write(b'0')  # poner servo en reposo
            self.ser.close()
        print(f"\n🔴 Captura detenida. Muestras registradas: {len(self.buffer_raw)}")

    def graficar(self):
        """Visualiza la sesión completa de captura."""
        if len(self.buffer_raw) < 10:
            print("⚠️ Pocas muestras para graficar. Captura al menos 2 segundos.")
            return

        t   = list(self.buffer_tiempo)
        raw = list(self.buffer_raw)
        rms = list(self.buffer_rms)

        color_cmd = {"REPOSO": "#95A5A6", "LEVE": "#F39C12", "FUERTE": "#E74C3C"}
        cmds      = list(self.buffer_cmds)

        fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
        fig.suptitle(f"Sesión BioAmp EXG Pill — {t[-1]:.1f}s capturados", fontsize=13, fontweight='bold')

        axes[0].plot(t, raw, color='#E74C3C', linewidth=0.7, alpha=0.85)
        axes[0].set_title("EMG crudo (desde BioAmp EXG Pill)", fontsize=9, fontweight='bold', loc='left')
        axes[0].set_ylabel("Voltaje (V)")
        axes[0].grid(True, alpha=0.3)
        axes[0].spines['top'].set_visible(False)
        axes[0].spines['right'].set_visible(False)

        axes[1].plot(t, rms, color='#8E44AD', linewidth=1.0)
        axes[1].axhline(self.umbral_leve,   color='#F39C12', linestyle='--', linewidth=1)
        axes[1].axhline(self.umbral_fuerte, color='#E74C3C', linestyle='--', linewidth=1)
        axes[1].set_title("Envolvente RMS + umbrales de decisión NERO", fontsize=9, fontweight='bold', loc='left')
        axes[1].set_xlabel("Tiempo (s)")
        axes[1].set_ylabel("RMS (V)")
        axes[1].grid(True, alpha=0.3)
        axes[1].spines['top'].set_visible(False)
        axes[1].spines['right'].set_visible(False)

        plt.tight_layout()
        plt.show()

        # Estadísticas finales
        from collections import Counter
        dist = Counter(cmds)
        print("\n📊 Distribución de activaciones:")
        for k, v in sorted(dist.items()):
            print(f"   {k:10s}: {v:4d} ({100*v/len(cmds):.1f}%)")


print("✅ Clase BioAmpNERO definida correctamente")
print("\n📋 Para usar con hardware real:")
print("   nero = BioAmpNERO(puerto='COM3')  # cambia el puerto")
print("   nero.iniciar()")
print("   time.sleep(15)  # captura 15 segundos")
print("   nero.detener()")
print("   nero.graficar()")


---
## Sección 6 — Demo completa sin hardware (modo simulado)

Si no tienes el Arduino disponible, este bloque simula el flujo completo:  
genera señal EMG → procesa → "envía" comandos simulados → grafica como si fuera real.

Útil para practicar el pipeline y entender los parámetros antes de conectar el hardware.


In [ ]:
# ── Simulación del flujo completo Arduino → Python ───────────────────────
print("🔄 Simulando sesión completa (modo sin hardware)...\n")

DURACION_SIM  = 12    # segundos
FS_SIM        = 100   # Hz (igual que Arduino)
UMBRAL_LEVE   = 0.05
UMBRAL_FUERTE = 0.12
VENTANA_N     = 20    # muestras (200 ms @ 100 Hz)

# Simular señal EMG a 100 Hz (resolución del Arduino)
emg_sim_100 = nk.emg_simulate(duration=DURACION_SIM, sampling_rate=FS_SIM,
                                burst_number=5, burst_duration=0.7, noise=0.01)

# Normalizar como lo haría el ADC de 10 bits
adc_sim = ((emg_sim_100 + 0.5) * 1023).clip(0, 1023).astype(int)

# ── Procesar muestra a muestra (igual que el hilo en tiempo real) ─────────
t_sim, raw_v, rms_v, cmds_v = [], [], [], []
buffer = deque(maxlen=VENTANA_N)

for i, adc in enumerate(adc_sim):
    voltaje = (adc / 1023.0) * 5.0 - 2.5  # reconstruir voltaje
    if buffer:
        voltaje_hp = voltaje - 0.95 * list(buffer)[-1]
    else:
        voltaje_hp = voltaje

    buffer.append(voltaje_hp)
    rms = np.sqrt(np.mean(np.array(buffer)**2)) if len(buffer) == VENTANA_N else 0.0

    cmd = 'FUERTE' if rms >= UMBRAL_FUERTE else ('LEVE' if rms >= UMBRAL_LEVE else 'REPOSO')

    t_sim.append(i / FS_SIM)
    raw_v.append(voltaje_hp)
    rms_v.append(rms)
    cmds_v.append(cmd)

# ── Visualización ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle("Simulación completa: BioAmp → Python → Arduino Uno",
             fontsize=13, fontweight='bold')

axes[0].plot(t_sim, raw_v, color='#E74C3C', linewidth=0.7)
axes[0].set_title("EMG simulado (equivalente a señal real del BioAmp EXG Pill)", fontsize=9, fontweight='bold', loc='left')
axes[0].set_ylabel("Voltaje (V)")
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].plot(t_sim, rms_v, color='#8E44AD', linewidth=1.0)
axes[1].axhline(UMBRAL_LEVE,   color='#F39C12', linestyle='--', linewidth=1, label='Umbral leve')
axes[1].axhline(UMBRAL_FUERTE, color='#E74C3C', linestyle='--', linewidth=1, label='Umbral fuerte')
axes[1].fill_between(t_sim, rms_v, UMBRAL_FUERTE,
                     where=[r > UMBRAL_FUERTE for r in rms_v], alpha=0.3, color='#E74C3C')
axes[1].fill_between(t_sim, rms_v, UMBRAL_LEVE,
                     where=[(UMBRAL_LEVE < r <= UMBRAL_FUERTE) for r in rms_v], alpha=0.3, color='#F39C12')
axes[1].set_title("Envolvente RMS y zonas de activación", fontsize=9, fontweight='bold', loc='left')
axes[1].set_ylabel("RMS (V)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

cmd_num = [0 if c == 'REPOSO' else (50 if c == 'LEVE' else 100) for c in cmds_v]
color_barra = ['#95A5A6' if c=='REPOSO' else ('#F39C12' if c=='LEVE' else '#E74C3C') for c in cmds_v]
axes[2].bar(t_sim, cmd_num, width=1/FS_SIM*0.9, color=color_barra, alpha=0.85)
axes[2].set_yticks([0, 50, 100])
axes[2].set_yticklabels(['0° Reposo', '50° Leve', '100° Fuerte'])
axes[2].set_title("Comando enviado al Arduino → Posición del servomotor", fontsize=9, fontweight='bold', loc='left')
axes[2].set_xlabel("Tiempo (s)")
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

from collections import Counter
dist = Counter(cmds_v)
print("\n📊 Resumen de la sesión simulada:")
for k, v in sorted(dist.items()):
    print(f"   {k:10s}: {v:4d} muestras ({100*v/len(cmds_v):.1f}%)")
print(f"\n✅ Pipeline completo ejecutado en modo simulado.")
print(f"   Con hardware real, reemplaza esta celda por: nero.iniciar()")


---
## Sección 7 — 🏆 Reto del taller

**Objetivo:** modificar el clasificador para que el servo tenga **5 posiciones** en lugar de 3.

```
Nivel 0: RMS < 0.03  → 0°   (reposo absoluto)
Nivel 1: RMS < 0.06  → 25°  (contracción muy leve)
Nivel 2: RMS < 0.09  → 50°  (contracción moderada)
Nivel 3: RMS < 0.12  → 75°  (contracción fuerte)
Nivel 4: RMS ≥ 0.12  → 100° (contracción máxima)
```

**Preguntas guía:**
1. ¿Cómo cambia el comportamiento del exoesqueleto con más niveles de control?
2. ¿Qué implicaciones tiene para pacientes de rehabilitación?
3. ¿Qué pasa si el umbral es muy bajo? ¿Y si es muy alto?

> 💡 **Bonus:** Implementa un filtro de histéresis para evitar que el servo "tiemble"  
> entre dos niveles cuando el músculo está justo en el umbral.


In [ ]:
# ── Tu implementación aquí ───────────────────────────────────────────────
# Modifica esta función para el clasificador de 5 niveles

def clasificador_5_niveles(rms_val):
    """
    Clasificador mejorado: 5 posiciones para el servo.
    
    Retorna: (angulo, etiqueta)
    """
    # ── TU CÓDIGO AQUÍ ──
    # Ejemplo de estructura:
    # if rms_val < 0.03:
    #     return 0, "REPOSO"
    # elif ...
    
    pass  # ← reemplaza esto


# ── Test del clasificador ─────────────────────────────────────────────────
valores_prueba = [0.01, 0.04, 0.07, 0.10, 0.15]

print("🧪 Prueba del clasificador de 5 niveles:")
print(f"{'RMS (V)':<12} {'Ángulo':<10} {'Etiqueta'}")
print("-" * 35)
for v in valores_prueba:
    resultado = clasificador_5_niveles(v)
    if resultado:
        angulo, etiqueta = resultado
        print(f"{v:<12.3f} {angulo:<10}° {etiqueta}")
    else:
        print(f"{v:<12.3f} {'pendiente':^10} (completar implementación)")


---
## ¡Taller completado! 🎉

### Resumen de lo aprendido

| Sección | Concepto | Librería/Tool |
|---------|----------|---------------|
| 1 | Señales biológicas simuladas (EMG, ECG, EEG) | NeuroKit2 |
| 2 | Pipeline EMG: filtrado → RMS → umbral | SciPy + NumPy |
| 3 | Análisis ECG, picos R y HRV | NeuroKit2 |
| 4 | Clasificador en ventana deslizante | NumPy |
| 5 | Integración con hardware real vía PySerial | PySerial |
| 6 | Demo completa simulada end-to-end | Todo lo anterior |

### Próximos pasos

- 🔬 **IPROTIC / UTP:** Unirse al grupo de investigación para trabajar en NERO v2
- 📄 **LACCEI 2026:** El taller conecta con la contribución #2624 en proceso
- 🏗️ **IEEE EMBS:** Rama estudiantil para continuar proyectos BCI
- 📦 **Repositorio:** Pedir acceso al código fuente de NERO al ponente

---
*Notebook desarrollado para el Taller IEEE EMBS × UTP — Ingeniería Biomédica con NERO*  
*Ponente: Rances Tello · IPROTIC / UTP Azuero*
